# P10 Network Analysis

Resource
* [Networkx Tutorial](https://networkx.org/documentation/stable/tutorial.html)
* [Intro to network analysis in python](https://trenton3983.github.io/files/projects/2020-05-21_intro_to_network_analysis_in_python/2020-05-21_intro_to_network_analysis_in_python.html) by Trenton McKinney (Recommended!)
* [Applied Graphical Network Analysis using Python](https://towardsdatascience.com/applied-network-analysis-using-python-25021633a702)
* [NetworkX:Network Analysis with Python](https://www.cl.cam.ac.uk/teaching/1415/L109/l109-tutorial_2015.pdf)
* [Network analysis for liquistics](https://alvinntnu.github.io/python-notes/statistical-analyses/network-analysis.html) by AlvinChen (Good!)
* [igraph with python](https://alvinntnu.github.io/python-notes/statistical-analyses/network-analysis-igraph.html) by Alvin Chen (Good!)
* [Text network analysis](https://towardsdatascience.com/text-network-analysis-generate-beautiful-network-visualisations-a373dbe183ca) towarddatascience (Good!)

In [2]:
import pandas as pd
raw = pd.read_csv("../_build/html/data/tweets20190315.csv")
raw

,tweetid,userid,user_display_name,user_screen_name,user_reported_location,user_profile_description,user_profile_url,follower_count,following_count,account_creation_date,...,latitude,longitude,quote_count,reply_count,like_count,retweet_count,hashtags,urls,user_mentions,poll_choices
0,1.118690e+18,828817384827875330,vdfxvgcvd8,vdfxvgcvd8,"Nasushiobara-shi, Tochigi",NaN,NaN,11401,11807,2017-02-07,...,absent,absent,0.0,0.0,0.0,0.0,[],[],[],NaN
1,1.107433e+18,769790067183190016,阿丽木琴,SamanthxBerg,NaN,我是一个小小的石头,NaN,13468,14606,2016-08-28,...,absent,absent,0.0,0.0,0.0,0.0,[],[],[],NaN
2,1.145785e+18,103804719,摩天轮约炮平台微信xdmtlove765,ronetaper,USA,Ronetaper,NaN,13268,61,2010-01-11,...,absent,absent,0.0,0.0,0.0,0.0,"['约炮', '护士', '国内交友', '二次元', '萝莉', '上海约炮', '北京约...",[],[],NaN
3,1.120716e+18,769790067183190016,阿丽木琴,SamanthxBerg,NaN,我是一个小小的石头,NaN,13468,14606,2016-08-28,...,absent,absent,0.0,0.0,0.0,0.0,[],[],[],NaN
4,1.114176e+18,769790067183190016,阿丽木琴,SamanthxBerg,NaN,我是一个小小的石头,NaN,13468,14606,2016-08-28,...,absent,absent,0.0,0.0,0.0,0.0,[],[],[],NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15574,1.145352e+18,2993185258,HK時政直擊,HKpoliticalnew,"Tsuen Wan District, Hong Kong","Love Hong Kong, love China.We should pay atten...",https://t.co/5u2Qw5PaKt,22551,66,2015-01-22,...,absent,absent,0.0,0.0,0.0,0.0,[],['https://news.rthk.hk/rthk/en/component/k2/14...,[213559194],NaN
15575,1.144128e+18,2993185258,HK時政直擊,HKpoliticalnew,"Tsuen Wan District, Hong Kong","Love Hong Kong, love China.We should pay atten...",https://t.co/5u2Qw5PaKt,22551,66,2015-01-22,...,absent,absent,0.0,0.0,0.0,0.0,"['HongKong', 'ExtraditionBill']",[],[213559194],NaN
15576,1.137938e+18,2993185258,HK時政直擊,HKpoliticalnew,"Tsuen Wan District, Hong Kong","Love Hong Kong, love China.We should pay atten...",https://t.co/5u2Qw5PaKt,22551,66,2015-01-22,...,absent,absent,0.0,0.0,0.0,0.0,[],[],[213559194],NaN
15577,1.139751e+18,2993185258,HK時政直擊,HKpoliticalnew,"Tsuen Wan District, Hong Kong","Love Hong Kong, love China.We should pay atten...",https://t.co/5u2Qw5PaKt,22551,66,2015-01-22,...,absent,absent,NaN,NaN,NaN,NaN,"['反送中', '香港']",[],[427607419],NaN


## Data to network

### from Pandas to network

In [72]:
df = raw[["userid", "user_mentions"]]
mentions = df.to_dict("records")
df.shape

(15579, 2)

In [76]:
df = df.assign(mention_len = lambda df:df.user_mentions.str.replace("\]|\[", "", regex=True).str.len())
df = df[df.mention_len > 0]
df

,userid,user_mentions,mention_len
10,828817384827875330,[10228272],8
438,828817384827875330,[10228272],8
579,903061563132616704,[10228272],8
601,1050020807798415365,[10228272],8
661,903061563132616704,[10228272],8
...,...,...,...
15574,2993185258,[213559194],9
15575,2993185258,[213559194],9
15576,2993185258,[213559194],9
15577,2993185258,[427607419],9


/var/folders/j3/p4x0mssx55nd8dn903h5wdb00000gn/T/ipykernel_65593/4207490096.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mention_len'] = df['user_mentions'].str.replace("\]|\[", "", regex=True).apply(lambda x:len(x))


In [61]:
df.user_mentions = df.user_mentions.str.replace("\]|\[", "", regex=True)

/var/folders/j3/p4x0mssx55nd8dn903h5wdb00000gn/T/ipykernel_65593/1048353317.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.user_mentions = df.user_mentions.str.replace("\]|\[", "", regex=True)


In [77]:
df

,userid,user_mentions,mention_len
10,828817384827875330,[10228272],8
438,828817384827875330,[10228272],8
579,903061563132616704,[10228272],8
601,1050020807798415365,[10228272],8
661,903061563132616704,[10228272],8
...,...,...,...
15574,2993185258,[213559194],9
15575,2993185258,[213559194],9
15576,2993185258,[213559194],9
15577,2993185258,[427607419],9


### from dict to network

In [78]:
df = raw[["userid", "user_mentions"]]
mentions = df.to_dict("records")

In [79]:
mentions[0].keys()

dict_keys(['userid', 'user_mentions'])

In [99]:
import re

edges = []
for relaiton in mentions:
    start, end = relaiton.values()
    end = re.sub("\[|\]", "", end)
    # end = end.replace("[|]", "")
    if end != "":
        if "," not in end:
            edges.append((start, end))
        if "," in end:
            for e in end.split(", "):
                edges.append((start, e))
edges[:10]

[('828817384827875330', '10228272'),
 ('828817384827875330', '10228272'),
 ('903061563132616704', '10228272'),
 ('1050020807798415365', '10228272'),
 ('903061563132616704', '10228272'),
 ('747289319384055808', '2201187373'),
 ('747274556994392064', '2201187373'),
 ('745784197617430528', '2353105641'),
 ('747289319384055808', '2201187373'),
 ('747289319384055808', '2201187373')]

In [102]:
from collections import Counter
edge_count = Counter(edges)
for edge, n in edge_count.most_common(10):
    print(edge, n)

('1194750901', '26257166') 171
('1368044863', '1142172626') 132
('1084444722', '25319414') 126
('1194750901', '2557521') 122
('4726278994', '10228272') 88
('1084444722', '26257166') 88
('746829685540143104', '1597435105') 87
('325103066', '1597435105') 77
('886933306599776257', '10228272') 76
('2968192342', '2957991374') 72


In [110]:
import pandas as pd
weighted_edges = [(edge[0], edge[1], n) for edge, n in edge_count.most_common()]
edges_df = pd.DataFrame(weighted_edges, columns=['v1','v2','sim'])
edges_df

,v1,v2,sim
0,1194750901,26257166,171
1,1368044863,1142172626,132
2,1084444722,25319414,126
3,1194750901,2557521,122
4,4726278994,10228272,88
...,...,...,...
3033,2993185258,25073877,1
3034,2993185258,20845447,1
3035,2993185258,1140572867882627078,1
3036,2993185258,824570238154862594,1


In [114]:
edges_df_short = edges_df.query('sim > 5')
edges_df_short

,v1,v2,sim
0,1194750901,26257166,171
1,1368044863,1142172626,132
2,1084444722,25319414,126
3,1194750901,2557521,122
4,4726278994,10228272,88
...,...,...,...
182,222444384,1604444052,6
183,1084444722,892761555745382402,6
184,222444384,148281099,6
185,1084444722,1021635608882507776,6


In [115]:
# !pip install pyvis
import networkx as nx
from pyvis.network import Network
G= nx.from_pandas_edgelist(edges_df_short, 'v1','v2','sim')

ModuleNotFoundError: No module named 'pyvis'